### Imports

In [ ]:
# ! pip install mlflow datasets langchain langchain-google-genai

In [1]:
import mlflow
import os
import json
from dotenv import load_dotenv
from datasets import load_dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from mlflow.genai.optimize import OptimizerConfig, LLMParams

from mlflow.genai import scorer

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "prompt_optimization"

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### Load the dataset and save 50 samples

#### Uncomment these cells when running for the first time

In [ ]:
# # Load AG News dataset from Hugging Face as pandas dataframe
# dataset = load_dataset("ag_news", split="train")
# df = dataset.to_pandas()

# df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

# df = df.sample(frac=1).reset_index(drop=True)

# df.head()

In [ ]:
# NUM_SAMPLES = 50
# train_data = []
# for i in range(NUM_SAMPLES):
#     article = df.iloc[i]["text"]
#     expected = df.iloc[i]["label"]
#     eval_dict = {
#         "inputs": {"article": article},
#         "expectations": {"expected_response": expected},
#     }
#     train_data.append(eval_dict)

# train_data[0]

In [ ]:
# # Save train_data list to a jsonl file
# import json
# with open("train_data.jsonl", "w") as f:
#     json.dump(train_data, f)

#### Load the training data from saved path

In [2]:
with open("train_data.jsonl", "r") as f:
    train_data = json.load(f)

train_data[0]

{'inputs': {'article': 'Productivity Growth Slows to 1.8 Percent  WASHINGTON (Reuters) - U.S. business productivity grew more  slowly in the third quarter than first thought, climbing at a  revised 1.8 percent annual rate, a government report showed on  Tuesday, while growth in unit labor costs was nudged up to a  1.8 percent pace.'},
 'expectations': {'expected_response': 'Business'}}

### Initialize the llm

In [3]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    rate_limiter=rate_limiter,
)

### Register an initial prompt

In [4]:
# Create a detailed prompt for classification
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/10/12 16:43:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 1


### Create a predict function and test with a sample

In [15]:
def predict_fn(article) -> str:
    prompt_template = mlflow.genai.load_prompt("news_classifier", version=1).template
    prompt = prompt_template.format(article=article)
    response = llm.invoke(prompt)
    return response.content

In [16]:
predict_fn(train_data[0]["inputs"]['article'])

'Business'

### Create an baseline

In [18]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations['expected_response']
    return outputs == expectations

results = mlflow.genai.evaluate(
    data=train_data,
    scorers=[exact_match],
    predict_fn=predict_fn,
)

2025/10/12 16:47:38 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/10/12 16:47:38 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Evaluating:   0%|          | 0/50 [Elapsed: 00:00, Remaining: ?] 

2025/10/12 16:47:49 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 16:47:49 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 16:47:49 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 16:47:49 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 16:47:49 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 16:47:49 WARNING mlflow.traci

### Optimize the prompt

In [21]:
# Automatically optimize the prompt using MLflow + DSPy

result = mlflow.genai.optimize_prompt(
    target_llm_params=LLMParams(model_name="openai/gpt-4o-mini"),
    prompt="prompts:/news_classifier/1",
    train_data=train_data[:40],
    scorers=[exact_match],
    eval_data=train_data[40:],  # Hold-out evaluation set
    optimizer_config=OptimizerConfig(
        num_instruction_candidates=10,  # Try 10 different prompt variations
        max_few_show_examples=10,  # Include up to 10 examples
    ),
)

2025/10/12 16:57:51 INFO mlflow.genai.optimize.base: 🚀 MLflow Run `bdb80250f9ed4d858c5ae2081546826e` started for prompt optimization! Watch the run to track the optimization progress.
2025/10/12 16:57:58 INFO mlflow.genai.optimize.optimizers.dspy_optimizer: 🎯 Starting prompt optimization for: prompts:/news_classifier/1
⏱️ This may take several minutes or longer depending on dataset size...
📊 Training with 40 examples.
2025/10/12 17:05:11 INFO mlflow.genai.optimize.optimizers.dspy_mipro_optimizer: Optimization complete! Score remained stable at: 0.0.
2025/10/12 17:05:11 INFO mlflow.genai.optimize.optimizers.dspy_mipro_optimizer: Optimization complete! Score remained stable at: 0.0.
2025/10/12 17:05:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 2


🏃 View run wistful-midge-517 at: http://localhost:5000/#/experiments/214623619727619851/runs/bdb80250f9ed4d858c5ae2081546826e
🧪 View experiment at: http://localhost:5000/#/experiments/214623619727619851


### Run the evaluation with Optimized prompt

In [5]:
def predict_fn(article) -> str:
    prompt_template = mlflow.genai.load_prompt("news_classifier", version=4).template
    prompt = prompt_template.format(article=article)
    response = llm.invoke(prompt)
    return response.content

In [6]:
sample_result = predict_fn(train_data[0]["inputs"]['article'])
print(sample_result)

```json
{
  "expected_response": "Business"
}
```


In [8]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations['expected_response']
    outputs = json.loads(outputs.replace("```json\n", "").replace("\n```", ""))["expected_response"]
    print(f"Outputs: {outputs}, Expectations: {expectations}")
    return outputs == expectations


with mlflow.start_run(run_name="optimized-prompt-eval"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

2025/10/12 17:45:58 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Evaluating:   0%|          | 0/50 [Elapsed: 00:00, Remaining: ?] 

2025/10/12 17:46:01 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


Outputs: Business, Expectations: Business


2025/10/12 17:46:03 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 17:46:03 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/12 17:46:03 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


...

Outputs: Sports, Expectations: Sports


2025/10/12 17:52:42 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


Outputs: World, Expectations: Science


2025/10/12 17:52:52 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


Outputs: Business, Expectations: Business
